In [29]:
import hashlib
import json
import pandas as pd
from pathlib import Path
from PIL import Image
from datetime import datetime, timezone

In [30]:

# 1. Dataset folder
folder = Path(
    r"C:\Users\samya\PyCharmProject\dataset-ind-vision-insp\screw_candidate_detection\candidate detection"
)


In [31]:
# Output folder — separate from your dataset
output_folder = Path(
    r"C:\Users\samya\PyCharmProject\industrial-vision-inspection\data"
)

output_folder.mkdir(parents=True, exist_ok=True)

image_csv_path = output_folder / "image_metadata.csv"
object_csv_path = output_folder / "object_annotations.csv"
image_extraction_errors = output_folder / "image_extraction_errors.csv"
annotation_extraction_errors = output_folder / "annotation_extraction_errors.csv"

In [32]:
#prepare lists to hold records
records = []
errors = []

In [33]:
# 4. Find every PNG image in the dataset folder
image_files = sorted(
    path for path in folder.glob("*.png")
    if path.parent != output_folder
)

print(f"PNG images found: {len(image_files)}")


PNG images found: 821


In [34]:
# 5. Process each image and its matching JSON
for image_path in image_files:
    json_path = image_path.with_suffix(".json")

    try:
        if not json_path.exists():
            raise FileNotFoundError(
                f"Matching JSON file not found: {json_path.name}"
            )

        # Read image properties
        with Image.open(image_path) as image:
            image_width, image_height = image.size
            image_format = image.format
            image_mode = image.mode
            channel_count = len(image.getbands())

        file_size_bytes = image_path.stat().st_size

        image_checksum = hashlib.sha256(
            image_path.read_bytes()
        ).hexdigest()

        # Read LabelMe JSON
        with json_path.open("r", encoding="utf-8") as file:
            annotation_data = json.load(file)

        shapes = annotation_data.get("shapes", [])

        if not isinstance(shapes, list):
            raise ValueError("'shapes' must be a list")

        json_width = annotation_data.get("imageWidth")
        json_height = annotation_data.get("imageHeight")

        # Validate dimensions against the real image
        if json_width != image_width or json_height != image_height:
            raise ValueError(
                f"Dimension mismatch: PNG={image_width}x{image_height}, "
                f"JSON={json_width}x{json_height}"
            )

        # Current UTC time
        now = datetime.now(timezone.utc).isoformat()

        # Build one image_metadata record
        record = {
            "image_id": image_path.stem,
            "image_filename": image_path.name,
            "image_format": image_format,
            "image_width_px": image_width,
            "image_height_px": image_height,
            "total_pixels": image_width * image_height,
            "file_size_bytes": file_size_bytes,
            "image_mode": image_mode,
            "channel_count": channel_count,
            "image_checksum_sha256": image_checksum,
            "labelme_version": annotation_data.get("version"),
            "json_filename": json_path.name,
            "json_image_width_px": json_width,
            "json_image_height_px": json_height,
            "total_annotations": len(shapes),
            "image_data_present": (
                annotation_data.get("imageData") is not None
            ),
            "annotation_status": "pending",
            "dataset_split": None,
            "created_at": now,
            "updated_at": now
        }

        records.append(record)

    except Exception as exc:
        errors.append({
            "image_filename": image_path.name,
            "json_filename": json_path.name,
            "error": str(exc)
        })


In [35]:
# 6. Create and save the combined CSV
columns = [
    "image_id",
    "image_filename",
    "image_format",
    "image_width_px",
    "image_height_px",
    "total_pixels",
    "file_size_bytes",
    "image_mode",
    "channel_count",
    "image_checksum_sha256",
    "labelme_version",
    "json_filename",
    "json_image_width_px",
    "json_image_height_px",
    "total_annotations",
    "image_data_present",
    "annotation_status",
    "dataset_split",
    "created_at",
    "updated_at"
]

image_metadata = pd.DataFrame(records, columns=columns)

# Detect duplicate IDs and checksums before saving
if not image_metadata.empty:
    duplicate_ids = image_metadata[
        image_metadata["image_id"].duplicated(keep=False)
    ]
    duplicate_checksums = image_metadata[
        image_metadata["image_checksum_sha256"].duplicated(keep=False)
    ]

    if not duplicate_ids.empty:
        raise ValueError(
            "Duplicate image IDs found:\n"
            + duplicate_ids["image_id"].to_string(index=False)
        )

    if not duplicate_checksums.empty:
        print(
            "Warning: identical image contents found under multiple "
            "filenames. Review the checksum uniqueness constraint."
        )

image_metadata.to_csv(image_csv_path, index=False)


In [36]:
# 7. Save errors separately
pd.DataFrame(
    errors,
    columns=["image_filename", "json_filename", "error"]
).to_csv(image_extraction_errors, index=False)

# 8. Report results
print("\nPROCESSING SUMMARY")
print("-" * 40)
print(f"Images discovered: {len(image_files)}")
print(f"Successfully processed: {len(records)}")
print(f"Errors: {len(errors)}")
print(f"Image_Metadata CSV: {image_csv_path}")
print(f"Errors CSV: {image_extraction_errors}")

display(image_metadata.head())

if errors:
    print("\nFiles requiring attention:")
    display(pd.DataFrame(errors))



PROCESSING SUMMARY
----------------------------------------
Images discovered: 821
Successfully processed: 821
Errors: 0
Image_Metadata CSV: C:\Users\samya\PyCharmProject\industrial-vision-inspection\data\image_metadata.csv
Errors CSV: C:\Users\samya\PyCharmProject\industrial-vision-inspection\data\image_extraction_errors.csv


,image_id,image_filename,image_format,image_width_px,image_height_px,total_pixels,file_size_bytes,image_mode,channel_count,image_checksum_sha256,labelme_version,json_filename,json_image_width_px,json_image_height_px,total_annotations,image_data_present,annotation_status,dataset_split,created_at,updated_at
0,00001,00001.png,PNG,1920,1080,2073600,1970722,RGB,3,73d866ee0dffd1b27b44daac1e1c4534196ecd62a7c5bd...,5.6.1,00001.json,1920,1080,19,False,pending,None,2026-10-17T17:27:18.977729+00:00,2026-10-17T17:27:18.977729+00:00
1,00002,00002.png,PNG,1920,1080,2073600,1950578,RGB,3,290ddaf57099e6e6c8488eb3eafda4c61e52b4d5a7c773...,5.6.1,00002.json,1920,1080,19,False,pending,None,2026-10-17T17:27:18.996391+00:00,2026-10-17T17:27:18.996391+00:00
2,00003,00003.png,PNG,1920,1080,2073600,1961830,RGB,3,ae9cc3623cdb3372c2fc6a5ca5b9c25b8d8ac950a0cb1c...,5.6.1,00003.json,1920,1080,22,False,pending,None,2026-10-17T17:27:19.015975+00:00,2026-10-17T17:27:19.015975+00:00
3,00004,00004.png,PNG,1920,1080,2073600,2138005,RGB,3,5859a9ea09311ec1e4b703683228a165549eb3ac4d6beb...,5.6.1,00004.json,1920,1080,20,False,pending,None,2026-10-17T17:27:19.036470+00:00,2026-10-17T17:27:19.036470+00:00
4,00005,00005.png,PNG,1920,1080,2073600,2026403,RGB,3,d0a480e723109c2c92167a26f69a08edd58dc6d152ff61...,5.6.1,00005.json,1920,1080,19,False,pending,None,2026-10-17T17:27:19.056354+00:00,2026-10-17T17:27:19.056354+00:00


In [37]:
obj_records = []
obj_errors = []

In [38]:
# 3. Find all JSON files directly inside the dataset folder
json_files = sorted(folder.glob("*.json"))

print(f"JSON files found: {len(json_files)}")

# Supported LabelMe shape types
supported_shape_types = {
    "rectangle", "polygon", "circle", "line",
    "point", "linestrip", "linestrip", "rotation"
}


JSON files found: 821


In [39]:

# 4. Process each JSON file
for json_path in json_files:
    image_id = json_path.stem
    image_path = folder / f"{image_id}.png"

    try:
        # The corresponding image is needed for dimensions
        if not image_path.exists():
            raise FileNotFoundError(
                f"Matching PNG not found: {image_path.name}"
            )

        with Image.open(image_path) as image:
            image_width, image_height = image.size

        with json_path.open("r", encoding="utf-8") as file:
            data = json.load(file)

        shapes = data.get("shapes", [])

        if not isinstance(shapes, list):
            raise ValueError("'shapes' must be a list")

        now = datetime.now(timezone.utc).isoformat()

        # 5. Create one record per shape
        for index, shape in enumerate(shapes, start=1):
            label = shape.get("label")
            shape_type = shape.get("shape_type")
            points = shape.get("points", [])
            flags = shape.get("flags") or {}

            if not label:
                raise ValueError(
                    f"Shape {index} has no label"
                )

            if not shape_type:
                raise ValueError(
                    f"Shape {index} has no shape_type"
                )

            if shape_type not in supported_shape_types:
                raise ValueError(
                    f"Unsupported shape_type '{shape_type}' "
                    f"in {json_path.name}, shape {index}"
                )

            if not isinstance(points, list) or len(points) == 0:
                raise ValueError(
                    f"Shape {index} has no coordinate points"
                )

            # Validate and convert coordinates
            coords = []
            for point in points:
                if not isinstance(point, (list, tuple)) or len(point) < 2:
                    raise ValueError(
                        f"Invalid point in shape {index}: {point}"
                    )

                x, y = float(point[0]), float(point[1])

                if not (pd.notna(x) and pd.notna(y)):
                    raise ValueError(
                        f"Invalid coordinate in shape {index}"
                    )

                coords.append((x, y))

            # Original LabelMe coordinates are retained in points_json.
            # Bounding box uses the extrema of all shape points.
            xs = [point[0] for point in coords]
            ys = [point[1] for point in coords]

            x1 = min(xs)
            y1 = min(ys)
            x2 = max(xs)
            y2 = max(ys)

            bbox_width = x2 - x1
            bbox_height = y2 - y1
            bbox_area = bbox_width * bbox_height

            center_x = (x1 + x2) / 2
            center_y = (y1 + y2) / 2

            # Normalized geometry
            x_center_norm = center_x / image_width
            y_center_norm = center_y / image_height
            width_norm = bbox_width / image_width
            height_norm = bbox_height / image_height
            area_norm = bbox_area / (image_width * image_height)

            aspect_ratio = (
                bbox_width / bbox_height
                if bbox_height > 0
                else None
            )

            # Out-of-bounds check
            out_of_bounds = (
                x1 < 0 or y1 < 0
                or x2 > image_width or y2 > image_height
            )

            annotation_id = f"image_{image_id}_{index:04d}"

            obj_records.append({
                "annotation_id": annotation_id,
                "image_id": image_id,
                "shape_index": index,
                "label": str(label),
                "shape_type": shape_type,
                "x1_px": x1,
                "y1_px": y1,
                "x2_px": x2,
                "y2_px": y2,
                "bbox_width_px": bbox_width,
                "bbox_height_px": bbox_height,
                "bbox_area_px2": bbox_area,
                "center_x_px": center_x,
                "center_y_px": center_y,
                "bbox_x_center_norm": x_center_norm,
                "bbox_y_center_norm": y_center_norm,
                "bbox_width_norm": width_norm,
                "bbox_height_norm": height_norm,
                "bbox_area_norm": area_norm,
                "bbox_aspect_ratio": aspect_ratio,
                "points_json": json.dumps(points),
                "annotation_flags": json.dumps(flags),
                "is_out_of_bounds": out_of_bounds,
                "annotation_review_status": "pending",
                "created_at": now
            })

    except Exception as exc:
        obj_errors.append({
            "json_filename": json_path.name,
            "image_filename": image_path.name,
            "error": str(exc)
        })




In [40]:
# 6. Create CSV with the full, fixed column structure
columns = [
    "annotation_id", "image_id", "shape_index", "label",
    "shape_type", "x1_px", "y1_px", "x2_px", "y2_px",
    "bbox_width_px", "bbox_height_px", "bbox_area_px2",
    "center_x_px", "center_y_px",
    "bbox_x_center_norm", "bbox_y_center_norm",
    "bbox_width_norm", "bbox_height_norm", "bbox_area_norm",
    "bbox_aspect_ratio", "points_json", "annotation_flags",
    "is_out_of_bounds", "annotation_review_status", "created_at"
]

object_annotations = pd.DataFrame(obj_records, columns=columns)



In [41]:
# 7. Check annotation ID uniqueness
if not object_annotations.empty:
    if object_annotations["annotation_id"].duplicated().any():
        raise ValueError("Duplicate annotation IDs found.")

    if object_annotations.duplicated(
        subset=["image_id", "shape_index"]
    ).any():
        raise ValueError("Duplicate image_id and shape_index pairs found.")

# 8. Save outputs
object_annotations.to_csv(object_csv_path, index=False)

pd.DataFrame(
    obj_errors,
    columns=["json_filename", "image_filename", "error"]
).to_csv(annotation_extraction_errors, index=False)

# 9. Summary
print("\nOBJECT ANNOTATIONS SUMMARY")
print("-" * 40)
print(f"JSON files discovered: {len(json_files)}")
print(f"Annotation records created: {len(object_annotations)}")
print(f"Files with errors: {len(obj_errors)}")
print(f"CSV saved to: {object_csv_path}")
print(f"Error report saved to: {annotation_extraction_errors}")

display(object_annotations.head())

if obj_errors:
    display(pd.DataFrame(obj_errors))



OBJECT ANNOTATIONS SUMMARY
----------------------------------------
JSON files discovered: 821
Annotation records created: 10226
Files with errors: 0
CSV saved to: C:\Users\samya\PyCharmProject\industrial-vision-inspection\data\object_annotations.csv
Error report saved to: C:\Users\samya\PyCharmProject\industrial-vision-inspection\data\annotation_extraction_errors.csv


,annotation_id,image_id,shape_index,label,shape_type,x1_px,y1_px,x2_px,y2_px,bbox_width_px,...,bbox_y_center_norm,bbox_width_norm,bbox_height_norm,bbox_area_norm,bbox_aspect_ratio,points_json,annotation_flags,is_out_of_bounds,annotation_review_status,created_at
0,image_00001_0001,00001,1,screw,rectangle,826.806723,86.638655,848.655462,106.806723,21.848739,...,0.089558,0.011380,0.018674,0.000213,1.083333,"[[826.8067226890755, 86.6386554621849], [848.6...",{},False,pending,2026-10-17T17:27:28.011994+00:00
1,image_00001_0002,00001,2,screw,rectangle,1111.680672,79.075630,1136.890756,100.084034,25.210084,...,0.082944,0.013130,0.019452,0.000255,1.200000,"[[1111.6806722689075, 79.07563025210086], [113...",{},False,pending,2026-10-17T17:27:28.011994+00:00
2,image_00001_0003,00001,3,screw,rectangle,1304.117647,195.882353,1319.243697,215.210084,15.126050,...,0.190321,0.007878,0.017896,0.000141,0.782609,"[[1304.1176470588234, 195.8823529411765], [131...",{},False,pending,2026-10-17T17:27:28.011994+00:00
3,image_00001_0004,00001,4,screw,rectangle,1411.680672,375.714286,1433.529412,396.722689,21.848739,...,0.357610,0.011380,0.019452,0.000221,1.040000,"[[1411.6806722689075, 375.7142857142857], [143...",{},False,pending,2026-10-17T17:27:28.011994+00:00
4,image_00001_0005,00001,5,screw,rectangle,1428.487395,590.000000,1450.336134,612.689076,21.848739,...,0.556800,0.011380,0.021008,0.000239,0.962963,"[[1428.487394957983, 590.0], [1450.33613445378...",{},False,pending,2026-10-17T17:27:28.011994+00:00


In [42]:
import pandas as pd

csv_path = r"C:\Users\samya\PyCharmProject\industrial-vision-inspection\data\object_annotations.csv"

df = pd.read_csv(csv_path)

print("OBJECT ANNOTATIONS VALIDATION")
print("-" * 40)

print("Total records:", len(df))
print("Unique annotation IDs:", df["annotation_id"].nunique())
print("Duplicate annotation IDs:", df["annotation_id"].duplicated().sum())
print("Missing image IDs:", df["image_id"].isna().sum())
print("Missing labels:", df["label"].isna().sum())
print("Missing shape types:", df["shape_type"].isna().sum())

print("\nLabels:")
print(df["label"].value_counts(dropna=False))

print("\nShape types:")
print(df["shape_type"].value_counts(dropna=False))

print("\nReview statuses:")
print(df["annotation_review_status"].value_counts(dropna=False))

print("\nOut-of-bounds annotations:")
print(df["is_out_of_bounds"].value_counts(dropna=False))

print("\nBounding-box checks:")
print("Negative widths:", (df["bbox_width_px"] < 0).sum())
print("Negative heights:", (df["bbox_height_px"] < 0).sum())
print("Zero-width boxes:", (df["bbox_width_px"] == 0).sum())
print("Zero-height boxes:", (df["bbox_height_px"] == 0).sum())

OBJECT ANNOTATIONS VALIDATION
----------------------------------------
Total records: 10226
Unique annotation IDs: 10226
Duplicate annotation IDs: 0
Missing image IDs: 0
Missing labels: 0
Missing shape types: 0

Labels:
label
screw    7941
hole     2285
Name: count, dtype: int64

Shape types:
shape_type
rectangle    10226
Name: count, dtype: int64

Review statuses:
annotation_review_status
pending    10226
Name: count, dtype: int64

Out-of-bounds annotations:
is_out_of_bounds
False    10226
Name: count, dtype: int64

Bounding-box checks:
Negative widths: 0
Negative heights: 0
Zero-width boxes: 0
Zero-height boxes: 0
